# **Data and Information Quality Project**

In [85]:
import pandas as pd
from ydata_profiling import ProfileReport
import json
import os
import re

In [86]:
DATASET = pd.read_csv('Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')


In [87]:
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,NaN,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,NaN,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,NaN,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,NaN,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,NaN,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN


# *Data Quality Assesement*


In [33]:
#Data information
DATASET.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3909 entries, 0 to 3908
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tipo esercizio pa      3878 non-null   object 
 1   Ubicazione             3909 non-null   object 
 2   Tipo via               3908 non-null   object 
 3   Via                    3908 non-null   object 
 4   Civico                 3832 non-null   object 
 5   Codice via             3908 non-null   float64
 6   ZD                     3908 non-null   object 
 7   Prevalente             294 non-null    object 
 8   Superficie altri usi   745 non-null    float64
 9   Superficie lavorativa  2601 non-null   float64
dtypes: float64(3), object(7)
memory usage: 305.5+ KB


##### Single column analisys

In [34]:
# Distinct values for each column
DATASET.nunique()

Tipo esercizio pa         103
Ubicazione               3554
Tipo via                   17
Via                      1370
Civico                    235
Codice via               1376
ZD                         10
Prevalente                 63
Superficie altri usi       52
Superficie lavorativa     145
dtype: int64

In [35]:
# Uniqueness percentage for each column
UNIQUENESS = (DATASET.nunique() / DATASET.shape[0]) * 100
UNIQUENESS

Tipo esercizio pa         2.634945
Ubicazione               90.918393
Tipo via                  0.434894
Via                      35.047327
Civico                    6.011768
Codice via               35.200819
ZD                        0.255820
Prevalente                1.611665
Superficie altri usi      1.330263
Superficie lavorativa     3.709389
dtype: float64

In [36]:
#Information about the type of esercizio
DATASET.value_counts("Tipo esercizio pa")

Tipo esercizio pa
Parrucchiere per signora                                         1048
ACCONCIATORE                                                      586
Parrucchiere per uomo                                             439
TIPO A - REG.2003                                                 335
TIPO A - REG.2003;TIPO B CENTRO DI ABBRONZATURA                   313
                                                                 ... 
TIPO A-B-C-D;ACCONCIATORE                                           1
TIPO A-B-C-D;Acconciatore                                           1
TIPO A-B-C-D;Estetista in profumeria                                1
TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ESTETICI DIMAGRIM       1
Truccatore                                                          1
Name: count, Length: 103, dtype: int64

##### Completeness

In [ ]:
# For each column
NULL_VALUES = DATASET.isnull().sum()
NOT_NULL_VALUES = DATASET.notnull().sum()
ROWS = DATASET.shape[0]
COMPLETENESS = NOT_NULL_VALUES / ROWS
COMPLETENESS = COMPLETENESS.map('{:.2%}'.format)
COMPLETENESS

Tipo esercizio             31
Ubicazione                  0
Tipo via                    1
Via                         1
Civico                     77
Codice via                  1
Municipio                   1
Attivita Primaria        3615
Superficie altri usi     3164
Superficie lavorativa    1308
tipo_via_check            149
indirizzo_check           149
civico_check              149
municipio_check           149
dtype: int64

In [38]:
# For the entire dataset
TOT_NULL_VALUES = DATASET.isnull().sum().sum()   # TODO: Controllare se abbiamo celle null con valori differenti
TOT_NOT_NULL_VALUES = DATASET.notnull().sum().sum()
TOT_COMPLETENESS = TOT_NOT_NULL_VALUES / DATASET.size
TOT_COMPLETENESS = '{:.2%}'.format(TOT_COMPLETENESS)
TOT_COMPLETENESS

'79.03%'

##### Duplication

In [39]:
DATASET.duplicated().any()

np.True_

In [40]:
DATASET[DATASET.duplicated()]

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
88,Acconciatore,VIA CORREGGIO N. 8 (z.d. 7),VIA,CORREGGIO,8,6287.0,7,ACCONCIATORE,NaN,NaN


# *Data Profiling*


In [41]:
profile = ProfileReport(DATASET, title=" Report comune di Milano Servizi alla persona di parrucchieri e estetisti")
profile.to_file("Report comune di Milano Servizi alla persona di parrucchieri e estetisti.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 31.40it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# *Data Wrangling*

Renaming and Sorting

In [42]:
DATASET.rename(columns={"Tipo esercizio pa":"Tipo esercizio", "Prevalente":"Attivita Primaria", "ZD":"Municipio"},inplace=True)

In [43]:
DATASET = DATASET.sort_values(by = ['Attivita Primaria', "Municipio"], ascending=True)
DATASET.head()

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa
95,Acconciatore,VIA DELLA MOSCOVA N. 48 (z.d. 1),VIA,DELLA MOSCOVA,48,1016.0,1,ACCONCIATORE,24.0,50.0
141,Acconciatore,VIA PARINI GIUSEPPE N. 9 (z.d. 1),VIA,PARINI GIUSEPPE,9,1049.0,1,ACCONCIATORE,15.0,47.0
164,Acconciatore,VIA SALA DEI LONGOBARDI N. 2 (z.d. 1),VIA,SALA DEI LONGOBARDI,2,217.0,1,ACCONCIATORE,NaN,NaN
220,ACCONCIATORE,CSO DI PORTA VITTORIA N. 32 ; (z.d. 1),CSO,DI PORTA VITTORIA,32,3016.0,1,ACCONCIATORE,NaN,49.0
229,ACCONCIATORE,CSO VENEZIA N. 8 ; (z.d. 1),CSO,VENEZIA,8,238.0,1,ACCONCIATORE,14.0,81.0


Standardization

In [44]:
def to_upper_safe(x):
    if isinstance(x, str):
        return x.upper()
    return x

DATASET = DATASET.map(lambda x: to_upper_safe(x) if pd.notnull(x) else x)

In [ ]:
# Transform "Tipo Esercizio"

Column Splitting

In [ ]:
# Split "Ubicazione"

# *Error Detection & Correction*

In [ ]:
# Fix "Ubicazione"

In [ ]:
# Maybe other fixes

# *Null Values Handling*

In [ ]:
# Drop rows where "Attivita Primaria" and "Tipo esercizio" are null

In [ ]:
# If "Tipo Esercizio" is NULL use "Attivita Primaria" top map it in the True/False columns 

In [ ]:
# If "Attivita Primaria" is null, fill with name of the first True column of "Tipo esercizio"

In [ ]:
# Fill "Superficie lavorativa" with median value grouping by "Tipo esercizio"

In [ ]:
# Fill "Superficie altri usi" with 0

In [ ]:
# Fill "Codice via" with values extracted from "Ubicazione" if existing, 0 otherwise

In [ ]:
# Fill "Municipio" with values extracted from "Ubicazione" if existing, 0 otherwise

# *Outlier Detection*

In [ ]:
# Compute Z-score on certain columns and trop outliers

# *Duplicate Detection*

In [ ]:
# Drop exact duplicates

In [ ]:
# Use 'Sorted Neighbourhood' to find possible duplicates

In [ ]:
# Drop new found possible duplicates